In [15]:
import sys

In [16]:
from pyspark.sql import SparkSession


In [17]:
spark = (SparkSession
        .builder
        .appName("PythonMnMCount")
        .getOrCreate())

Get the M&M data set file name

The dataset contains 100,000 lines, with each line being <state, color, count>. We want to compute and aggregate the counts for each color and state.

In [18]:
mnm_file = "data/mnm_dataset.csv"

In [19]:
mnm_df = (spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(mnm_file))

In [20]:
mnm_df.show(n=5, truncate=False)

+-----+-----+-----+
|State|Color|Count|
+-----+-----+-----+
|WY   |Brown|61   |
|ID   |Brown|34   |
|OR   |Green|47   |
|WY   |Red  |93   |
|ID   |Blue |94   |
+-----+-----+-----+
only showing top 5 rows


Aggregate count of all colors and groupBy state and color

orderBy descending order

In [21]:
count_mnm_df = (mnm_df.select("State", "Color", "Count")
                    .groupBy("State", "Color")
                    .sum("Count")
                    .orderBy("sum(Count)", ascending=False))

In [22]:
count_mnm_df.show(n=60, truncate=False)

+-----+------+----------+
|State|Color |sum(Count)|
+-----+------+----------+
|UT   |Blue  |97437     |
|WA   |Red   |95552     |
|UT   |Yellow|95333     |
|WA   |Brown |95258     |
|WY   |Blue  |95185     |
|OR   |Orange|94506     |
|ID   |Yellow|94441     |
|NV   |Orange|94338     |
|UT   |Brown |94242     |
|CO   |Red   |94177     |
|OR   |Blue  |94110     |
|WY   |Green |94105     |
|ID   |Green |94057     |
|AZ   |Orange|94039     |
|NM   |Brown |94015     |
|OR   |Red   |93896     |
|OR   |Yellow|93516     |
|WA   |Blue  |93253     |
|CA   |Brown |93183     |
|AZ   |Blue  |93062     |
|CA   |Red   |92788     |
|AZ   |Yellow|92787     |
|UT   |Red   |92727     |
|WA   |Orange|92722     |
|ID   |Brown |92579     |
|CA   |Blue  |92560     |
|CO   |Orange|92523     |
|NM   |Green |92478     |
|NM   |Red   |92366     |
|CA   |Orange|92312     |
|NM   |Yellow|92285     |
|WY   |Yellow|92116     |
|ID   |Blue  |91965     |
|OR   |Green |91775     |
|NV   |Yellow|91690     |
|WY   |Orang

In [23]:
print("Total Rows = %d" % (count_mnm_df.count()))

Total Rows = 60


The above code aggergated and counted for all the states. What if we just want to see the data for a single state. To do that: 
 1. Select from all rows in the DataFrame
 2. Filter only one state
 3. groupBy() State and Color as we did before
 4. Aggregate the counts for each color
 5. orderBy() in descending order
    

In [26]:
  ca_count_mnm_df = (mnm_df.select("*")
                       .where(mnm_df.State == 'ID')
                       .groupBy("State", "Color")
                       .sum("Count")
                       .orderBy("sum(Count)", ascending=False))

Show the resulting aggregation for California

In [27]:
ca_count_mnm_df.show(n=10, truncate=False)

+-----+------+----------+
|State|Color |sum(Count)|
+-----+------+----------+
|ID   |Yellow|94441     |
|ID   |Green |94057     |
|ID   |Brown |92579     |
|ID   |Blue  |91965     |
|ID   |Orange|88168     |
|ID   |Red   |86919     |
+-----+------+----------+



26/10/08 01:08:10 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 972485 ms exceeds timeout 120000 ms
26/10/08 01:08:10 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/08 01:08:14 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora